In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from loguru import logger

# ML lab 06 -  Gradient descent for logistic regression

### Implement gradient descent for:

- learning the weights of a logistic regression model using the logistic sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$ as activation function
- by minimization of the cross-entropy (log-loss)

Useful facts for this are:

- $\mathbf{\hat{y}} = \sigma[\mathbf{X}\mathbf{w}]$
- $E(\mathbf{w}) = - \sum_{i=1}^n \left[ y_i \log(\hat{y_i}) + (1-y_i) \log(1 - \hat{y_i})\right] = - (\mathbf{y}^T log(\mathbf{\hat{y}}) + (1-\mathbf{y})^T log(1-\mathbf{\hat{y}}))$
- $\nabla E(\mathbf{w}) = \mathbf{X}^T \left(\mathbf{y} - \mathbf{\hat{y}} \right)$


## Instructions for this lab are:

1. Implement gradient descent for logistic regression using formulas above, with L1 and/or L2 regularization; you can use the skeleton below that uses three hyperparameters: 
    - the _learning rate_ 
    - the regularization parameter $\lambda$, and
    - the max number of iterations of gradient descent

    The code provided is just a suggestion, feel free to write your own style if you feel more comfortable. 
    Notice that the formulas given above are for unregularized logistic regression, so you will have to add the bit that is missing reflecting the regularization.

2. Experiment with different initializations and hyperparameter settings to see which combination gives you the best cross-validation accuracy (or use any resampling method of your choice)

3. Do this for the heart data from the fifth lab so that you can directly compare results obtained from sklearn functions and your gradient descent implementation


In [8]:

# sigmoid function, argument can be scalar or array
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, _max_iter = 100, _lr = 0.1, _lambda=0, _alpha=0.5,_regularize=False):
    """ implements _batch_ gradient descent for logistic regression for a binary classification problem (uses log-loss or binary cross-entropy)
        parameters are:
        X: feature matrix
        y: labels
        _max_iter: max number of iterations
        _lr: learning rate
        _lambda: regularization strength
        _regularize: if False, no regularization, if "l1" then lasso regularization, if "l2" then ridge regularization
    """

    # prepend column of 1s to X to allow for bias
    # 
    X0 = np.ones((X.shape[0],1))
    X = np.hstack((X0,X))

    # proceed
    n, d = X.shape
    # INITIALIZATION STRATEGY FILL ALL WEIGHTS WITH 0
    w = np.zeros(shape=(d))
    y_hat = sigmoid(X @ w)  # prediction with initial weights
    err = [- (y.T @ np.log(y_hat) + (1-y).T @ np.log(1- y_hat))]   # error with initial weights (log loss)

    for t in range(_max_iter):

        general_grad = (X.T @ (y_hat - y)) / n

        # compute gradient
        if _regularize == 'l1': # lasso
            grad = general_grad + _lambda * np.sign(w)
        elif _regularize == 'l2': # ridge
            grad = general_grad + _lambda * np.square(w)
        elif _regularize == 'elasticnet':
            grad = general_grad + _lambda * (_alpha * np.square(w) + (1-_alpha) * np.sign(w))
        else:  # no regularization
            grad = general_grad 


        # update weights
        w = w - _lr * grad
        # This is lacking bias?!?!?!?
        
        # obtain new predictions
        y_hat = sigmoid(X @ w)

        # current loss (averaged 1/n)
        loss =  - np.sum(y.T @ np.log(y_hat) + (1-y).T @ np.log(1-y_hat))/n

        # quit if converged.....
        if np.isclose(loss, err[-1]):
            break
        err.append(loss)

    print(f'finished after {t+1} iterations.')

    return w, err

## The heart dataset

In [9]:
heart = pd.read_csv("heart.csv", delimiter=',')
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols = ['sex', 'cp', 'exang',  'fbs', 'restecg', 'slope', 'ca', 'thal']
target = 'target'
heart[categorical_cols] = heart[categorical_cols].astype('category')

heart.describe(include='all')

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.000000,303.0,303.0,303.000000,303.000000,303.0,303.0,303.000000,303.0,303.000000,303.0,303.0,303.0,303.000000
unique,NaN,2.0,4.0,NaN,NaN,2.0,3.0,NaN,2.0,NaN,3.0,5.0,4.0,NaN
top,NaN,1.0,0.0,NaN,NaN,0.0,1.0,NaN,0.0,NaN,2.0,0.0,2.0,NaN
freq,NaN,207.0,143.0,NaN,NaN,258.0,152.0,NaN,204.0,NaN,142.0,175.0,166.0,NaN
mean,54.366337,NaN,NaN,131.623762,246.264026,NaN,NaN,149.646865,NaN,1.039604,NaN,NaN,NaN,0.544554
std,9.082101,NaN,NaN,17.538143,51.830751,NaN,NaN,22.905161,NaN,1.161075,NaN,NaN,NaN,0.498835
min,29.000000,NaN,NaN,94.000000,126.000000,NaN,NaN,71.000000,NaN,0.000000,NaN,NaN,NaN,0.000000
25%,47.500000,NaN,NaN,120.000000,211.000000,NaN,NaN,133.500000,NaN,0.000000,NaN,NaN,NaN,0.000000
50%,55.000000,NaN,NaN,130.000000,240.000000,NaN,NaN,153.000000,NaN,0.800000,NaN,NaN,NaN,1.000000
75%,61.000000,NaN,NaN,140.000000,274.500000,NaN,NaN,166.000000,NaN,1.600000,NaN,NaN,NaN,1.000000


In [12]:
#PREPROCESSING PIPELINE

# Drop missing values
heart_imputed = heart.dropna()

# One-Hot Encode categorical columns
heart_encoded = pd.get_dummies(heart, columns=categorical_cols, drop_first=True)

# Separate Features and Target
X = heart_encoded.drop(columns=[target])
y = heart_encoded[target]

# Convert to NumPy arrays
X = X.to_numpy()
y = y.to_numpy()

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [13]:
model, errors = gradient_descent(X_train, y_train)

TypeError: loop of ufunc does not support argument 0 of type float which has no callable exp method